# 03c · Test-image bootstrap for the de-duplicated label-efficiency comparisons (CPU, ~2 min)

Run **after 03b**. The paired t-test in 03b measures variation across the 20 label subsets only; it treats the test images as fixed. On the 539-image de-duplicated rice test set, test-image sampling is the larger source of uncertainty.

For every domain checkpoint against ImageNet DINO, at each label budget, this notebook:
1. averages each test image's correctness over the 20 subsets;
2. takes the per-image difference between the two models;
3. resamples test images 2,000 times (identically for both models) to get a 95% percentile CI.

This is the paper's primary effect-size method, applied to subset-averaged predictions. Send back `03c_le_dedup_bootstrap.json`.

In [ ]:
import os, sys, json
import numpy as np, pandas as pd
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive; drive.mount('/content/drive')
BASE = os.environ.get('SSL_BASE', '/content/drive/MyDrive/SSL_AgriStudy')
R = f'{BASE}/results_v2'; OUT_C = f'{R}/correct/le'
drop_sets = json.load(open(f'{R}/14_drop_sets.json'))
TARGETS = ['RiceLeaf', 'FloraBD', 'Flowers102', 'CUB']
DOMAIN  = ['florabd_full_scratch', 'florabd_cont_in', 'rice_full', 'rice_cont_in', 'dinov2_vits14']
DROPS   = ['none', 'cos0.95', 'cos0.95+phash8']
rows = []
for t in TARGETS:
    ref = np.load(f'{OUT_C}/imagenet_dino__{t}.npz'); y = ref['y']; keys = list(ref['keys'])
    fracs = sorted({float(k.split('__s')[0]) for k in keys})
    for bb in DOMAIN:
        z = np.load(f'{OUT_C}/{bb}__{t}.npz'); assert list(z['keys']) == keys and (z['y'] == y).all()
        for frac in fracs:
            sel = [i for i, k in enumerate(keys) if float(k.split('__s')[0]) == frac]
            a = (z['pred'][sel] == y).mean(0); b = (ref['pred'][sel] == y).mean(0)   # per-image correctness, subset-averaged
            for d in DROPS:
                keep = np.setdiff1d(np.arange(len(y)), np.array(drop_sets[t][d], int))
                diff = 100 * (a[keep] - b[keep])
                rng = np.random.RandomState(0)
                boot = diff[rng.randint(0, len(keep), (2000, len(keep)))].mean(1)
                lo, hi = np.percentile(boot, [2.5, 97.5])
                rows.append(dict(target=t, backbone=bb, fraction=frac, dropset=d, n_test=int(len(keep)),
                                 n_subsets=len(sel), diff=float(diff.mean()), ci_lo=float(lo), ci_hi=float(hi),
                                 ci_excludes_zero=bool(lo > 0 or hi < 0)))
res = pd.DataFrame(rows)
json.dump(rows, open(f'{R}/03c_le_dedup_bootstrap.json', 'w'), indent=1)
print('saved', f'{R}/03c_le_dedup_bootstrap.json')
show = res[res.target == 'RiceLeaf'].copy()
show['CI'] = show.apply(lambda r: f"{r['diff']:+.2f} [{r.ci_lo:+.1f}, {r.ci_hi:+.1f}]", axis=1)
display(show.pivot_table(index=['backbone', 'fraction'], columns='dropset', values='CI', aggfunc='first')[DROPS])

In [ ]:
if IN_COLAB:
    from google.colab import drive; drive.flush_and_unmount(); print('✓ uploaded to Drive')